# Setup

In [ ]:
import json, os, re
from pathlib import Path
from typing import Literal

import anthropic
import jinja2
import yaml
from dotenv import load_dotenv
from pydantic import BaseModel

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

load_dotenv(REPO / ".env", override=True)

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "No ANTHROPIC_API_KEY in the environment. "
        "Copy .env.example to .env, set the key, then re-run this cell."
    )

# The model is pinned per phase in prompts/phases/*.yaml (see PHASE below), so one
# phase can be moved to a different model without touching this notebook.
client = anthropic.Anthropic()

print("client ready")

# Judging a submission


In [ ]:
# The rubric lives in criteria/problem_definition.yaml, so this notebook, the judge,
# and CLive Studio (`uv run clive-studio`) all read the same one. Edit it there and
# re-run this cell rather than editing a copy here.
CRITERIA = yaml.safe_load((REPO / "criteria/problem_definition.yaml").read_text())["criteria"]

print(f"{len(CRITERIA)} criteria: {', '.join(c['id'] for c in CRITERIA)}")

## 2. Verdict

In [9]:
class Verdict(BaseModel):
    criterion_id: str
    verdict: Literal["PASS", "FAIL"]
    evidence: str
    confidence: Literal["low", "medium", "high"]


class JudgeResult(BaseModel):
    verdicts: list[Verdict]


# This mirrors prompts/base/output_schema.json -- keep the two in step.
print(json.dumps(JudgeResult.model_json_schema(), indent=2)[:400], "...")

{
  "$defs": {
    "Verdict": {
      "properties": {
        "criterion_id": {
          "title": "Criterion Id",
          "type": "string"
        },
        "verdict": {
          "enum": [
            "PASS",
            "FAIL"
          ],
          "title": "Verdict",
          "type": "string"
        },
        "evidence": {
          "title": "Evidence",
          "type": "string"
       ...


## 3. Rendering the prompt

In [ ]:
PHASE = yaml.safe_load((REPO / "prompts/phases/problem_definition.yaml").read_text())
PROBLEM = yaml.safe_load((REPO / "cases/problems/grade_average.yaml").read_text())

_jinja = jinja2.Environment(trim_blocks=True, lstrip_blocks=True, undefined=jinja2.Undefined)


def render_user_prompt(problem: dict, artifact: dict, criteria: list[dict], attempt: int = 1) -> str:
    """Render the phase template. `artifact` is the student's submission."""
    return _jinja.from_string(PHASE["user_template"]).render(
        problem=problem,
        artifact=artifact,
        # The STUDENT ARTIFACT block loops over the phase's declared fields, so a
        # field added in CLive Studio reaches the prompt with no template edit.
        artifact_fields=PHASE["artifact_fields"],
        criteria_to_judge=criteria,
        attempt=attempt,
    )

good_artifact = {
    "summary": (
        "I need to work out the class's mean score. The program reads how many students "
        "there are, then reads each student's mark, adds them up and divides by the count."
    ),
    "inputs": "int n (the student count), then n floats, one score per student",
    "outputs": "a single float: the mean of the n scores, printed to two decimal places",
}

prompt = render_user_prompt(PROBLEM, good_artifact, CRITERIA)
print(prompt)

## 4. The judge call

One call judges every criterion at once. `output_format=JudgeResult` is what turns the reply into
a validated object instead of text — `resp.parsed_output` comes back as a `JudgeResult`, already
type-checked.

Note what is *absent*: no `temperature`. Deliberation is bought with `effort` and adaptive
thinking instead, which lets the model reason before committing to a verdict.

In [ ]:
def judge(problem: dict, artifact: dict, criteria: list[dict], attempt: int = 1) -> JudgeResult:
    """Judge one submission against every criterion in a single call."""
    try:
        resp = client.messages.parse(
            model=PHASE["model"]["id"],
            max_tokens=PHASE["model"]["max_output_tokens"],
            system=PHASE["system_prompt"],
            messages=[
                {"role": "user", "content": render_user_prompt(problem, artifact, criteria, attempt)}
            ],
            thinking={"type": "adaptive"},
            output_config={"effort": PHASE["model"]["effort"]},
            output_format=JudgeResult,
        )
    except anthropic.AuthenticationError:
        raise RuntimeError(
            "No valid API key. Copy .env.example to .env and set ANTHROPIC_API_KEY."
        ) from None
    except anthropic.NotFoundError:
        raise RuntimeError(
            f"Model {PHASE['model']['id']!r} is not available to this key."
        ) from None
    except anthropic.RateLimitError as e:
        retry_after = e.response.headers.get("retry-after", "60")
        raise RuntimeError(f"Rate limited; retry after {retry_after}s.") from None
    except anthropic.APIStatusError as e:
        raise RuntimeError(f"API error {e.status_code}: {e.message}") from None
    except anthropic.APIConnectionError:
        raise RuntimeError("Could not reach the API. Check your connection.") from None

    if resp.stop_reason == "max_tokens":
        raise RuntimeError(
            "Response hit max_tokens and the verdict list is truncated. "
            "Raise max_output_tokens in the phase YAML, or lower `effort`."
        )

    return resp.parsed_output


result = judge(PROBLEM, good_artifact, CRITERIA)

for v in result.verdicts:
    print(f"{v.verdict:4}  {v.criterion_id:18} ({v.confidence})")
    print(f"        evidence: {v.evidence!r}\n")